In [ ]:
from pyspark.sql import SparkSession, Row
from pyspark.ml import PipelineModel
from pyspark.ml.classification import LogisticRegressionModel, RandomForestClassificationModel
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.sql.functions import col, udf, when
from pyspark.sql.types import DoubleType

# 1. INITIALIZE SPARK
spark = SparkSession.builder.appName("NYC_Taxi_Stage8_Ultimate").config("spark.driver.memory", "8g").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# 2. LOAD HELD-OUT TEST DATA (Rubric: Touched only at this stage)
print("Loading completely unseen Test data...")
test_df = spark.read.parquet("../data/processed/test.parquet")
pipeline_model = PipelineModel.load("../models/preprocessing_pipeline")
test_prepared = pipeline_model.transform(test_df)

# 3. LOAD BOTH MODELS
print("Loading Both Classifier Models...")
lr_model = LogisticRegressionModel.load("../models/best_logistic_regression")
rf_model = RandomForestClassificationModel.load("../models/best_random_forest")

preds_lr = lr_model.transform(test_prepared)
preds_rf = rf_model.transform(test_prepared)

# 4. COMPARE MODELS ON A SINGLE DATAFRAME (Rubric requirement)
bin_eval_roc = BinaryClassificationEvaluator(labelCol="has_tip", metricName="areaUnderROC")
bin_eval_pr = BinaryClassificationEvaluator(labelCol="has_tip", metricName="areaUnderPR")
multi_eval_acc = MulticlassClassificationEvaluator(labelCol="has_tip", metricName="accuracy")

resultados = [
    Row(Model="Logistic Regression", 
        ROC_AUC=bin_eval_roc.evaluate(preds_lr), 
        PR_AUC=bin_eval_pr.evaluate(preds_lr), 
        Accuracy=multi_eval_acc.evaluate(preds_lr)),
    Row(Model="Random Forest", 
        ROC_AUC=bin_eval_roc.evaluate(preds_rf), 
        PR_AUC=bin_eval_pr.evaluate(preds_rf), 
        Accuracy=multi_eval_acc.evaluate(preds_rf))
]
results_df = spark.createDataFrame(resultados)
print("\n=== MODEL COMPARISON DATAFRAME ===")
results_df.show()

# (Random Forest won, so we use preds_rf for the rest of the metrics)
print("\n=== PER-CLASS METRICS (Random Forest) ===")
# Rubric: precision, recall, F1, accuracy per class
for clase in [0.0, 1.0]:
    print(f"--- Clase {int(clase)} (0=No Tip, 1=Tip) ---")
    precision = MulticlassClassificationEvaluator(labelCol="has_tip", metricLabel=clase, metricName="precisionByLabel").evaluate(preds_rf)
    recall = MulticlassClassificationEvaluator(labelCol="has_tip", metricLabel=clase, metricName="recallByLabel").evaluate(preds_rf)
    f1 = MulticlassClassificationEvaluator(labelCol="has_tip", metricLabel=clase, metricName="fMeasureByLabel").evaluate(preds_rf)
    print(f"Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}")

# 5. CONFUSION MATRIX
print("\n=== CONFUSION MATRIX ===")
preds_rf.groupBy("has_tip", "prediction").count().orderBy("has_tip", "prediction").show()

# 6. ROC & PR CURVE POINTS (Rubric: extracted from model's summary)
# We evaluate the Logistic Regression model on the Test dataset to extract the native summary from Spark
print("\n=== ROC AND PR CURVE POINTS (Sample of 5) ===")
lr_test_summary = lr_model.evaluate(test_prepared)

print("PR Curve Points (Recall vs Precision):")
lr_test_summary.pr.show(5)

print("ROC Curve Points (FPR vs TPR):")
lr_test_summary.roc.show(5)

# 7. THRESHOLD SWEEP & DOMAIN CONSTRAINT
print("\n=== THRESHOLD SWEEP ANALYSIS ===")
extract_prob = udf(lambda v: float(v[1]), DoubleType())
pred_probs = preds_rf.withColumn("prob_tip", extract_prob(col("probability")))

# Sweep evaluating Precision and Recall for thresholds 0.5 vs 0.3
for th in [0.5, 0.3]:
    temp_preds = pred_probs.withColumn("custom_pred", when(col("prob_tip") >= th, 1.0).otherwise(0.0))
    tp = temp_preds.filter((col("custom_pred") == 1) & (col("has_tip") == 1)).count()
    fp = temp_preds.filter((col("custom_pred") == 1) & (col("has_tip") == 0)).count()
    fn = temp_preds.filter((col("custom_pred") == 0) & (col("has_tip") == 1)).count()
    
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    print(f"Threshold {th}: Recall = {rec:.4f}, Precision = {prec:.4f}")

# 8. ERROR ANALYSIS VIA SPARK SQL
print("\n=== ERROR ANALYSIS ON SPARK SQL ===")
errors_df = preds_rf.filter(col("has_tip") != col("prediction"))
errors_df.createOrReplaceTempView("model_errors")

print("SQL Query 1: Do errors happen more often at specific hours?")
spark.sql("SELECT pickup_hour, COUNT(*) as error_count FROM model_errors GROUP BY pickup_hour ORDER BY error_count DESC LIMIT 5").show()

print("SQL Query 2: Which fare distances confuse the model the most?")
spark.sql("""
    SELECT 
        CASE 
            WHEN trip_distance < 2 THEN 'Short (<2m)'
            WHEN trip_distance BETWEEN 2 AND 5 THEN 'Medium (2-5m)'
            ELSE 'Long (>5m)' END as distance_cat,
        COUNT(*) as error_count
    FROM model_errors
    GROUP BY CASE WHEN trip_distance < 2 THEN 'Short (<2m)' WHEN trip_distance BETWEEN 2 AND 5 THEN 'Medium (2-5m)' ELSE 'Long (>5m)' END
""").show()

print("Stage 8 completed! Perfect Rubric Match.")

26/05/27 02:27:27 WARN Utils: Your hostname, LaptopPablo resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/05/27 02:27:27 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/05/27 02:27:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Loading completely unseen Test data...


Loading Both Classifier Models...


...

Threshold 0.3: Recall = 1.0000, Precision = 0.9652

=== ERROR ANALYSIS ON SPARK SQL ===
SQL Query 1: Do errors happen more often at specific hours?



[Stage 155:================================>                       (8 + 6) / 14]



+-----------+-----------+
|pickup_hour|error_count|
+-----------+-----------+
|         18|       2083|
|         23|       2054|
|         20|       2050|
|         19|       2048|
|         22|       2039|
+-----------+-----------+

SQL Query 2: Which fare distances confuse the model the most?



[Stage 158:================================>                       (8 + 6) / 14]



+-------------+-----------+
| distance_cat|error_count|
+-------------+-----------+
|Medium (2-5m)|      11634|
|   Long (>5m)|       3542|
|  Short (<2m)|      22698|
+-------------+-----------+

Stage 8 completed! Perfect Rubric Match.
